In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

device = torch.device("cuda:4") if torch.cuda.is_available() else torch.device("cpu")
print(f"Working with {device}")

Working with cuda:4


In [4]:
class MLP(nn.Module):
    def __init__(self, num_inputs, num_hiddens, num_outputs, num_layers):
        super().__init__()
        self.input_layer = nn.Linear(num_inputs, num_hiddens)        
        hidden_layers = []
        for _ in range(num_layers):
            hidden_layers.append(nn.Linear(num_hiddens, num_hiddens))
        self.hidden_layers = nn.ModuleList(hidden_layers)
        self.output_layer = nn.Linear(num_hiddens, num_outputs)

    def forward(self, x):
        x = F.silu(self.input_layer(x))
        for layer in self.hidden_layers:
            x = F.silu(layer(x))
        x = F.log_softmax(self.output_layer(x), -1)
        return x

In [5]:
from torchvision import datasets
from torch.utils.data import DataLoader


class FastMNIST(datasets.MNIST):
    def __init__(self, *args, **kwargs):
        num_data = kwargs.pop("num_data", None)
        super().__init__(*args, **kwargs)

        self.data = self.data.float().div(255).reshape(-1, 784)
        if num_data is not None:
            self.data, self.targets = self.data[:num_data], self.targets[:num_data]
        self.data, self.targets = self.data.to(device), self.targets.to(device)

    def __getitem__(self, index):
        return self.data[index], self.targets[index]


class FastFashionMNIST(datasets.FashionMNIST):
    def __init__(self, *args, **kwargs):
        num_data = kwargs.pop("num_data", None)
        super().__init__(*args, **kwargs)

        self.data = self.data.float().div(255).reshape(-1, 784)
        if num_data is not None:
            self.data, self.targets = self.data[:num_data], self.targets[:num_data]
        self.data, self.targets = self.data.to(device), self.targets.to(device)

    def __getitem__(self, index):
        return self.data[index], self.targets[index]


num_train = 5000
train_batch_size = 100
test_batch_size = 1000

training_data = FastMNIST(root="./data", train=True, download=True, num_data=num_train)
test_data = FastMNIST(root="./data", train=False, download=True)
train_dataloader = DataLoader(
    training_data, batch_size=train_batch_size, shuffle=True, num_workers=0
)
test_dataloader = DataLoader(test_data, batch_size=test_batch_size, num_workers=0)

ood_data = FastFashionMNIST(root="./data", download=True, train=False)
ood_dataloader = DataLoader(ood_data, batch_size=test_batch_size, num_workers=0)


100.0%
100.0%
100.0%
100.0%
100.0%
100.0%
100.0%
100.0%


In [7]:
import torch.optim as optim
from evaluation import accuracy

num_inputs = 784
num_outputs = 10
num_hiddens = 128
num_layers = 2


def run_sgdm(lr=5.0e-2, weight_decay=1.0e-3):

    model = MLP(num_inputs, num_hiddens, num_outputs, num_layers).to(device)

    opt = optim.SGD(model.parameters(), lr, momentum=0.9, weight_decay=weight_decay)

    num_epochs = 100
    train_num_batches = len(train_dataloader)
    test_num_batches = len(test_dataloader)
    total_num_steps = num_epochs * train_num_batches
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, total_num_steps)

    for epoch in range(1, num_epochs + 1):

        model.train()
        train_acc = 0.0
        train_nll = 0.0
        for x, y in train_dataloader:
            opt.zero_grad()
            pred_logits = model(x)
            loss = F.cross_entropy(pred_logits, y)
            loss.backward()
            opt.step()
            scheduler.step()

            train_acc += accuracy(pred_logits, y).item()
            train_nll += loss.item()

        train_acc /= train_num_batches
        train_nll /= train_num_batches

        if epoch % 20 == 0:

            model.eval()
            with torch.no_grad():
                test_acc = 0.0
                test_nll = 0.0
                for x, y in test_dataloader:
                    pred_logits = model(x)
                    test_acc += accuracy(pred_logits, y)
                    test_nll += F.cross_entropy(pred_logits, y)

            test_acc /= test_num_batches
            test_nll /= test_num_batches

            print(
                f"Epoch {epoch}  lr {scheduler.get_last_lr()[0]:.4f} "
                f"train: nll {train_nll:.4f} acc {train_acc:.4f} "
                f"test: nll {test_nll:.4f} acc {test_acc:.4f}"
            )

    return model.state_dict()

In [8]:
samples = []
num_samples = 6
for i in range(1, num_samples + 1):
    print(f"training model {i}...")
    samples.append(run_sgdm())

training model 1...
Epoch 20  lr 0.0452 train: nll 0.0401 acc 0.9870 test: nll 0.2802 acc 0.9298
Epoch 40  lr 0.0327 train: nll 0.0076 acc 1.0000 test: nll 0.2829 acc 0.9330
Epoch 60  lr 0.0173 train: nll 0.0054 acc 1.0000 test: nll 0.2591 acc 0.9381
Epoch 80  lr 0.0048 train: nll 0.0059 acc 1.0000 test: nll 0.2602 acc 0.9373
Epoch 100  lr 0.0000 train: nll 0.0059 acc 1.0000 test: nll 0.2588 acc 0.9374
training model 2...
Epoch 20  lr 0.0452 train: nll 0.0338 acc 0.9920 test: nll 0.2791 acc 0.9310
Epoch 40  lr 0.0327 train: nll 0.0071 acc 0.9998 test: nll 0.2738 acc 0.9353
Epoch 60  lr 0.0173 train: nll 0.0062 acc 1.0000 test: nll 0.2583 acc 0.9371
Epoch 80  lr 0.0048 train: nll 0.0062 acc 1.0000 test: nll 0.2540 acc 0.9381
Epoch 100  lr 0.0000 train: nll 0.0061 acc 1.0000 test: nll 0.2532 acc 0.9377
training model 3...
Epoch 20  lr 0.0452 train: nll 0.0351 acc 0.9910 test: nll 0.2585 acc 0.9340
Epoch 40  lr 0.0327 train: nll 0.0063 acc 0.9998 test: nll 0.2562 acc 0.9393
Epoch 60  lr 0

In [9]:
from evaluation import evaluate

model = MLP(num_inputs, num_hiddens, num_outputs, num_layers).to(device)
pred_logits = []
ood_pred_logits = []

with torch.no_grad():
    for sample in samples:
        model.load_state_dict(sample)
        pred_logits_ = []
        for x, _ in test_dataloader:
            pred_logits_.append(model(x))
        pred_logits.append(torch.cat(pred_logits_, 0))

        ood_pred_logits_ = []
        for x, _ in ood_dataloader:
            ood_pred_logits_.append(model(x))
        ood_pred_logits.append(torch.cat(ood_pred_logits_, 0))

pred_logits = torch.stack(pred_logits, 0)
ood_pred_logits = torch.stack(ood_pred_logits, 0)

metrics = evaluate(pred_logits, ood_pred_logits, test_data.targets)
print(f"individual models")
print(
    f"acc {metrics['indiv']['acc']:.4f} nll {metrics['indiv']['nll']:.4f} "
    f"ece {metrics['indiv']['ece']:.4f} auroc {metrics['indiv']['auroc']:.4f}"
)
print("ensemble model")
print(
    f"acc {metrics['ens']['acc']:.4f} nll {metrics['ens']['nll']:.4f} "
    f"ece {metrics['ens']['ece']:.4f} auroc {metrics['ens']['auroc']:.4f}"
)

individual models
acc 0.9378 nll 0.2544 ece 0.0296 auroc 0.8063
ensemble model
acc 0.9403 nll 0.2217 ece 0.0222 auroc 0.8547
